# EEG Source-Space Fronto-Parietal Theta Connectivity

**Author:** Cynthia Deng

**Dataset:** Aging and Cognitive Control, Stimulus-Locked EEG

---

## Goal

The goal of this pipeline is to compute fronto-parietal theta-band connectivity from stimulus-locked EEG epochs, using source-space imaging rather than sensor-space signals.

Specifically, I am building a pipeline that:

1. Takes preprocessed, epoched EEG data (.set files from EEGLAB) from participants in an aging and cognitive control study
2. Projects the scalp EEG signals into source space using a template brain (fsaverage), so that connectivity is measured between brain regions rather than electrodes
3. Computes ciPLV (corrected imaginary Phase-Locking Value) in the theta band (5 to 7 Hz) between frontal and parietal regions of interest (ROIs)
4. Produces a per-subject fronto-parietal connectivity score for each condition (ontask and offtask), ready for statistical analysis

**Why source space?**
Scalp EEG connectivity is heavily contaminated by volume conduction, which means the same neural source can appear at many electrodes and create spurious synchrony. By projecting to source space first, we localize activity to anatomically defined brain regions before computing connectivity.

**Why ciPLV?**
ciPLV measures phase synchrony between two signals while being mathematically immune to zero-lag volume conduction artifacts. It is well suited for studying long-range cortical communication, such as the fronto-parietal theta synchrony associated with attention and working memory.

**Why theta (5 to 7 Hz)?**
Frontal theta oscillations are consistently linked to top-down cognitive control, and fronto-parietal theta synchrony is proposed as a mechanism for coordinating attention networks, which are known to change with aging.

---

## Template Overview

I started this project with a general-purpose template (`eeg_connectivity_template_python.py`) for computing fronto-parietal ciPLV from EEGLAB epochs using MNE and mne-connectivity. The template gave me a solid starting structure that covered all the core steps of a source connectivity pipeline:

```
Load fsaverage anatomy (source space + BEM)
        |
        v
Read atlas labels -> define frontal + parietal ROIs
        |
        v
For each subject:
    Load EEGLAB epochs
    Set montage + average reference
    Filter to theta band
    Estimate noise covariance
    Build forward model
    Build inverse operator
    Apply inverse -> source estimates
    Extract ROI time series
    Compute ciPLV (Fourier, 4 to 7 Hz)
        |
        v
Return ciPLV matrices + ROI names
```

The template was written to be general and applicable to any dataset with BioSemi 128-channel EEG. It used reasonable defaults throughout. My job was to understand each part of it, and then adapt it to the specific needs of this aging study.

---

## My Decisions

After working through the template and understanding each part, I made the following changes to adapt it to this dataset and research question.

**1. Inverse method: eLORETA instead of dSPM**
The template used dSPM, which works well but tends to make sources near the surface of the brain look stronger than they actually are. For this study, where I am comparing brain network activity across age groups, that bias could quietly distort the results. I switched to eLORETA, which treats surface and deep sources equally.

**2. Adaptive lambda2 instead of a fixed value**
The template assumed every subject's data has the same quality by using a fixed regularization value. Signal quality varies a lot in an aging sample, so I compute this value from each subject's actual data instead.

**3. pca_flip instead of mean_flip for ROI time series**
The template averaged all the source points within a region to get one time series. I use pca_flip instead, which finds the single best representation of what all the points in that region are doing together. This is more robust, especially for larger regions.

**4. Narrower theta band: 5 to 7 Hz instead of 4 to 7 Hz**
At 4 Hz, only about 2 full cycles fit inside our 600 ms connectivity window, which is not enough for a reliable phase estimate. Starting at 5 Hz ensures at least 3 full cycles fit, giving the connectivity measure something solid to work with.

**5. Stimulus-locked connectivity window (0.0 to 0.6 s)**
The template computed connectivity across the full epoch. I restrict it to 0.0 to 0.6 seconds post-stimulus to focus on the window where active cognitive processing is happening.

**6. Surrogate resampling for trial-count balancing**
The template had no trial balancing. I added 500 surrogate draws, each using a fixed number of trials, to make connectivity scores comparable across subjects regardless of how many clean epochs they had.

**7. Hemisphere-specific ROI indexing**
The template combined all ROI labels without tracking which hemisphere they belonged to. I separate left and right hemisphere labels explicitly so I can report LH and RH fronto-parietal connectivity independently.

**8. Handling BioSemi external channels (EXG1 to EXG8)**
BioSemi files include extra non-scalp channels that the template did not account for. I mark them as misc so MNE excludes them from source imaging.

**9. Global forward solution computed once**
The template recomputed the forward solution for every subject. Since all subjects share the same channel layout and template brain, I compute it once and reuse it.

**10. Multitaper for point estimate, Fourier for surrogates**
For the final connectivity estimate I use multitaper, which is more stable. For the 500 surrogate draws I use Fourier, which is faster, since stability is already achieved by averaging across draws.

---

## Step-by-Step Pipeline

---

### Step 0 -- Imports and Configuration

**What it does:**
This cell loads all the tools the notebook needs, and sets every adjustable parameter in one place at the top. Things like which subject to run, which condition, what time windows to use, and how many surrogate draws to do are all defined here.

**Why we do this:**
Having all the settings in one place means you never have to dig through the code to change something. If you want to run a different subject or swap the condition from ontask to offtask, you change it here and everything else updates automatically. It also makes the notebook easier for anyone else to read, since all the analysis choices are visible right at the top.

In [ ]:
import numpy as np
import mne
from mne_connectivity import spectral_connectivity_epochs
from pathlib import Path
import warnings
import json
import pandas as pd

# -------------------------------------------------------
# CONFIGURATION -- update these to match your setup
# -------------------------------------------------------

"EEGLAB_DIR  = Path("../data")  # folder containing .set files
"
SUBJECT_ID  = 301                          # subject to run
"CONDITION   = "offtask"                    # ontask or offtask
"
OUT_DIR     = Path("./output")
OUT_DIR.mkdir(exist_ok=True)

# ---- Frequency
# Narrowed from template's 4-7 Hz to ensure at least 3 full cycles in the 600 ms window
THETA_BAND = (5., 7.)

# ---- Epoch and connectivity windows
TMIN, TMAX   = -0.2, 0.8   # full epoch crop window in seconds
COV_TMIN     = -0.18        # noise covariance baseline start
COV_TMAX     = -0.02        # noise covariance baseline end
CONN_TMIN    = 0.0          # connectivity window starts at stimulus onset
CONN_TMAX    = 0.6          # connectivity window ends at 600 ms
BASELINE_WIN = (-0.2, 0.0)  # pre-stimulus window used as noise for SNR
RESP_WIN     = (0.0, 0.8)   # response window used as signal for SNR

# ---- Surrogate resampling
N_SURROGATES = 500   # number of random trial draws
SUBSAMPLE_N  = 94    # trials per draw -- set to the lowest trial count across all subjects
RANDOM_STATE = 0     # fixed seed for reproducibility

# ---- Source imaging
SPACING        = "oct6"     # source space resolution
INVERSE_METHOD = "eLORETA" # chosen over dSPM for zero depth bias

rng = np.random.default_rng(RANDOM_STATE)

---

### Step 1 -- Load Shared Anatomy (fsaverage)

**What it does:**
This step loads a standard brain template called fsaverage, which is a kind of average brain built from many MRI scans. From it, we build three things:

- A source space -- a map of thousands of points on the cortical surface where we will estimate brain activity
- A head model (BEM) -- a 3-layer representation of the skull, brain, and scalp that describes how electrical signals travel through the head
- A transformation file -- a small file that aligns the electrode positions on the scalp to the brain template's coordinate system

**Why we do this:**
Since we do not have individual MRI scans for each participant, we use this shared template brain for everyone. These three objects are built once and reused for every subject, which saves a lot of time. They are the anatomical foundation that makes source imaging possible. Without them, we have no way to go from scalp electrodes to brain regions.

In [ ]:
# Download fsaverage to MNE's local data directory (skipped if already present)
fs_subjects_dir = mne.datasets.fetch_fsaverage(verbose=True)
subjects_dir    = str(Path(fs_subjects_dir).parent)
subject         = "fsaverage"

# Transformation file: aligns sensor positions to the fsaverage MRI coordinate system
trans = Path(subjects_dir) / subject / "bem" / "fsaverage-trans.fif"

# Source space: a grid of candidate dipole locations on the cortical surface
# add_dist=False skips distance computation between sources, which saves time
src = mne.setup_source_space(
    subject,
    spacing=SPACING,
    subjects_dir=subjects_dir,
    add_dist=False
)

# BEM model: 3-layer head model representing brain, skull, and scalp
# conductivity values (S/m) are standard values used in the field
bem_model = mne.make_bem_model(
    subject=subject,
    ico=4,
    conductivity=(0.3, 0.006, 0.3),
    subjects_dir=subjects_dir
)

# BEM solution: precomputes the basis needed to build the forward model
bem = mne.make_bem_solution(bem_model)

print(f"Source space: {src}")
print(f"BEM solution ready.")

---

### Step 2 -- Define Fronto-Parietal ROIs

**What it does:**
This step uses a brain atlas to identify specific regions of interest on the cortical surface. Think of the atlas as a labeled map of the brain that divides the cortex into named regions. We pick six regions total:

- 3 frontal regions -- areas in the prefrontal cortex involved in attention and cognitive control
- 3 parietal regions -- areas in the parietal cortex involved in attention and sensory processing

**Change from template -- hemisphere tracking:**
The template grabbed all matching labels from both hemispheres and put them into one combined list, without keeping track of which side they came from. For this study, I explicitly separate the left and right hemisphere versions of each region. This means at the end we can report connectivity for each hemisphere independently.

**Why we do this:**
We are not interested in connectivity between all possible brain regions, just the fronto-parietal network, which is the specific network linked to the cognitive processes this study is about. Keeping hemispheres separate gives us more detailed results. LH and RH fronto-parietal connectivity can behave differently, especially in aging, and collapsing them together would hide that.

In [ ]:
# Load all parcellation labels from the Desikan-Killiany atlas
# Each label is a named cortical region, for example 'superiorfrontal-lh'
labels = mne.read_labels_from_annot(
    subject,
    parc="aparc",
    subjects_dir=subjects_dir
)

frontal_rois  = {"rostralmiddlefrontal", "caudalmiddlefrontal", "superiorfrontal"}
parietal_rois = {"superiorparietal", "inferiorparietal", "supramarginal"}

# Build a lookup dictionary keyed by (region_name, hemisphere)
# rsplit splits on the last hyphen only, safely handling region names that contain hyphens
label_dict = {}
for lab in labels:
    region, hemi = lab.name.rsplit("-", 1)
    label_dict[(region, hemi)] = lab

# Helper to retrieve labels for a set of regions in a given hemisphere
def pick_labels(regions, hemi):
    return [label_dict[(r, hemi)] for r in regions if (r, hemi) in label_dict]

# Separate by hemisphere -- needed for LH and RH connectivity later
frontal_lh  = pick_labels(frontal_rois,  "lh")
frontal_rh  = pick_labels(frontal_rois,  "rh")
parietal_lh = pick_labels(parietal_rois, "lh")
parietal_rh = pick_labels(parietal_rois, "rh")

# Validate that all four groups have labels
if any(len(x) == 0 for x in [frontal_lh, frontal_rh, parietal_lh, parietal_rh]):
    raise RuntimeError("One or more ROI groups are empty -- check atlas label names.")

# Final label order is [F_lh, F_rh, P_lh, P_rh]
# This order defines the rows and columns of the connectivity matrix
fp_labels = frontal_lh + frontal_rh + parietal_lh + parietal_rh
roi_names = [lab.name for lab in fp_labels]
n_rois    = len(fp_labels)

# Index arrays for each group -- used later to slice the connectivity matrix by hemisphere
n_flh, n_frh = len(frontal_lh), len(frontal_rh)
n_plh, n_prh = len(parietal_lh), len(parietal_rh)

idx_flh = np.arange(0, n_flh)
idx_frh = np.arange(n_flh, n_flh + n_frh)
idx_plh = np.arange(n_flh + n_frh, n_flh + n_frh + n_plh)
idx_prh = np.arange(n_flh + n_frh + n_plh, n_rois)

print(f"Total ROIs: {n_rois}")
print(f"ROI names: {roi_names}")

---

### Step 3 -- Load and Preprocess EEG Epochs

**What it does:**
This step loads the actual EEG data for a subject -- the stimulus-locked epochs that were already cleaned and prepared in EEGLAB. Then it does a few preparation steps before source imaging can happen:

- Removes non-scalp channels -- BioSemi systems record a few extra channels like eye movement electrodes that are not brain signals. We tell MNE to ignore them
- Sets electrode positions -- attaches the known 3D positions of each electrode on the scalp so the software knows where each sensor is physically located
- Applies an average reference -- a standard correction that removes any bias introduced by the choice of reference electrode
- Crops the epoch -- trims the data to just the time window we care about

**Change from template -- handling non-scalp channels:**
The template assumed all channels in the file were clean scalp EEG and skipped this step entirely. In practice, BioSemi 128-channel files always include 8 external channels that are not brain signals. If left in, they cause errors when fitting the electrode positions and corrupt the source imaging. I mark them as misc so MNE knows to exclude them.

**Change from template -- global forward solution:**
The template recomputed the forward solution separately for every subject. Since all subjects share the same electrode layout and we are using the same template brain, this was unnecessary. I compute it once from the first subject and reuse it for everyone, which saves significant time.

**Why we do this:**
The EEG data as loaded is not yet ready for source imaging. The software needs to know where every electrode sits on the scalp, the data needs to be in a reference-neutral form, and any channels that do not belong need to be excluded. These steps make sure the data is clean and correctly formatted before we start projecting it into the brain.

In [ ]:
# File naming convention for this dataset
file_pattern = f"sub-{{sub}}_erpcon_stimlocked_{CONDITION}.set"

# BioSemi external channels -- not scalp EEG, must be excluded before montage fitting
not_scalp = ["EXG1", "EXG2", "EXG3", "EXG4", "EXG5", "EXG6", "EXG7", "EXG8"]

montage = mne.channels.make_standard_montage("biosemi128")

# Load epochs for this subject
fname  = EEGLAB_DIR / file_pattern.format(sub=SUBJECT_ID)
epochs = mne.io.read_epochs_eeglab(fname, verbose=False)

# Mark external channels as misc so MNE excludes them from source imaging
present = [ch for ch in not_scalp if ch in epochs.ch_names]
if present:
    epochs.set_channel_types({ch: "misc" for ch in present})
    print(f"Marked as misc: {present}")

# Crop to analysis window before referencing to save memory
epochs.crop(tmin=TMIN, tmax=TMAX)

# Set electrode positions -- on_missing='ignore' skips any unmatched channels
epochs.set_montage(montage, on_missing="ignore")

# Average reference: removes the bias from whichever reference electrode was used
# Applied as a projection so it is tracked and can be reversed if needed
epochs.set_eeg_reference("average", projection=True)
epochs.apply_proj()

n_total = len(epochs)
sfreq   = epochs.info["sfreq"]
ep_tmin = epochs.tmin

print(f"Loaded {n_total} epochs | tmin={epochs.tmin:.3f}  tmax={epochs.tmax:.3f}")
print(f"Sampling rate: {sfreq} Hz")

---

### Step 4 -- Compute Global Forward Solution

**What it does:**
This step builds a mathematical map that describes how a signal originating from any point in the brain would look at each scalp electrode. It combines the electrode positions from Step 3 with the head model and source space from Step 1 to create this map. It is computed once using the first subject's electrode layout and then reused for all subjects.

**Why we do this:**
Before we can work backwards from scalp signals to brain sources, we first need to understand the forward direction. If the brain fires here, what does the scalp see? This is a necessary building block for the next step. Because all subjects in this study use the same 128-channel BioSemi setup and the same template brain, this map is identical for everyone, so there is no need to recompute it per subject.

In [ ]:
# Forward solution: maps source dipoles to scalp electrode voltages
# Computed once using the current subject's channel layout (same for all subjects)
fwd_global = mne.make_forward_solution(
    epochs.info,       # channel positions and layout
    trans=str(trans),  # coordinate transform aligning sensor space to MRI space
    src=src,           # source space (cortical dipole locations)
    bem=bem,           # BEM solution (volume conductor model)
    eeg=True,          # we are working with EEG, not MEG
    mindist=5.0,       # exclude sources within 5 mm of inner skull for numerical stability
    verbose=False
)

print(f"Forward solution: {fwd_global}")

---

### Step 5 -- Noise Covariance, Inverse Operator, and Adaptive Lambda2

**What it does:**
This step works backwards from the scalp signals to estimate what was happening in the brain, which is the reverse of Step 4. It has three parts:

- Noise covariance -- looks at the pre-stimulus period before anything happened to get a sense of what the background noise in the signal looks like
- Inverse operator -- combines the forward solution and the noise estimate to build a tool that can translate scalp voltages into brain source activity
- Adaptive lambda2 -- a tuning parameter that controls how much the solution trusts the data versus smoothing it out

**Change from template -- adaptive lambda2:**
The template used a fixed value for lambda2, essentially assuming that every subject's data has the same quality. In reality, signal quality varies a lot across participants, especially in an aging sample where older adults often have noisier EEG. I compute lambda2 from each subject's actual data by comparing how strong the signal is during the response window versus the background noise in the pre-stimulus baseline. Subjects with cleaner data get a value that trusts the data more, and subjects with noisier data get more smoothing applied. This makes the pipeline fairer and better adapted to the real variability in the dataset.

**Why we do this:**
Going from scalp to brain is mathematically ambiguous since many different brain activity patterns could produce the same scalp signal. The inverse operator is our best mathematical solution to that problem, and lambda2 is how we handle uncertainty in that solution. Computing it from the real data rather than assuming a fixed value makes the results more reliable across subjects.

In [ ]:
# Noise covariance from pre-stimulus baseline
# 'shrunk' method applies shrinkage regularization for a stable estimate
noise_cov = mne.compute_covariance(
    epochs,
    tmin=COV_TMIN,
    tmax=COV_TMAX,
    method="shrunk",
    verbose=False
)

# Inverse operator
# loose=0.2 allows a small amount of tangential dipole orientation
# depth=0.8 prevents the solution from over-favoring superficial sources
inverse_op = mne.minimum_norm.make_inverse_operator(
    epochs.info,
    fwd_global,
    noise_cov,
    loose=0.2,
    depth=0.8
)

# Compute adaptive lambda2 from median epoch SNR
# RMS helper: computes signal power within a given time window
def rms(ep, win):
    w0, w1 = win
    s = int(round((w0 - ep_tmin) * sfreq))  # convert time in seconds to sample index
    e = int(round((w1 - ep_tmin) * sfreq))
    return np.sqrt(np.mean(ep[:, s:e] ** 2))

snrs = []
for ep in epochs.get_data():
    sig = rms(ep, RESP_WIN)           # signal: power in the response window
    noi = rms(ep, BASELINE_WIN)       # noise: power in the pre-stimulus baseline
    snrs.append(sig / (noi + 1e-12))  # small value added to avoid division by zero

median_snr = np.median(snrs)
lambda2    = 1.0 / (median_snr ** 2)

print(f"Median SNR:  {median_snr:.3f}")
print(f"Lambda2:     {lambda2:.6f}")
print(f"  (template fixed value was 1/9 = {1/9:.6f})")

---

### Step 6 -- Source Reconstruction and ROI Time Series Extraction

**What it does:**
This step applies the inverse operator to the actual EEG data. For every epoch and every time point, it estimates what brain activity was happening at each point on the cortical surface. Then it summarizes that activity down to one time series per ROI, so instead of thousands of individual brain points we end up with 12 time series, one per ROI across both hemispheres.

**Change from template -- eLORETA instead of dSPM:**
The template used a method called dSPM. It works well but has a known tendency to make sources near the surface of the brain look stronger than they actually are. This is called depth bias. For this study, where we are comparing brain network activity across age groups, that bias could quietly distort the results. I switched to eLORETA, which is specifically designed to treat surface and deep sources equally, giving a fairer picture of where activity is actually coming from.

**Change from template -- pca_flip instead of mean_flip:**
Once we have source activity, each ROI contains dozens of individual source points. We need to collapse those into one representative time series. The template used mean_flip, which simply averages them after aligning their signs. I use pca_flip instead, which finds the single direction that best captures what all the points in the region are doing together. This is a smarter summary that holds up better when the region is large or the points within it are not all doing exactly the same thing.

**Why we do this:**
This is the core of source imaging. We are converting scalp EEG into estimated brain activity, localized to the specific regions we care about. The output of this step is what the connectivity analysis in the next step actually runs on. Everything up to this point has been preparation to get here.

In [ ]:
# Apply inverse operator to all epochs
# Returns a list of SourceEstimate objects, one per epoch
stcs = mne.minimum_norm.apply_inverse_epochs(
    epochs,
    inverse_op,
    lambda2=lambda2,          # adaptive regularization computed from median SNR
    method=INVERSE_METHOD,    # eLORETA -- no depth bias
    pick_ori="normal",        # constrain dipoles to surface-normal orientation
    verbose=False
)

# Extract one time series per ROI per epoch
# pca_flip finds the first principal component across all dipoles in the label
# return_generator=False computes everything at once so we can convert to a numpy array
label_ts_all = mne.extract_label_time_course(
    stcs,
    fp_labels,
    inverse_op["src"],
    mode="pca_flip",
    return_generator=False
)

# Convert to numpy array with shape (n_epochs, n_rois, n_times)
label_ts_all = np.asarray(label_ts_all)

print(f"label_ts_all shape: {label_ts_all.shape}")
print(f"  -> (n_epochs={label_ts_all.shape[0]}, n_rois={label_ts_all.shape[1]}, n_times={label_ts_all.shape[2]})")

---

### Step 7 -- ciPLV Connectivity (Point Estimate)

**What it does:**
This step takes the 12 ROI time series from Step 6 and computes connectivity between every pair of regions. It is asking how synchronized these two brain regions are in the theta frequency range. The result is a matrix where every cell represents how connected two ROIs are.

**Change from template -- multitaper instead of Fourier:**
The template used a method called Fourier to estimate the frequency content of the signal. It works, but produces noisier estimates, especially when the time window is short as ours is at 600 ms. I switched to multitaper, which runs multiple slightly different versions of the same analysis and averages them together. The result is a more stable and reliable connectivity estimate. The trade-off is that it is a little slower, but for the final reported values that stability is worth it.

**Change from template -- narrower theta band (5 to 7 Hz instead of 4 to 7 Hz):**
At 4 Hz, a single cycle takes 250 ms, meaning only about 2 full cycles fit inside our 600 ms window. That is not enough to get a reliable phase estimate. By starting at 5 Hz instead, we ensure at least 3 full cycles fit in the window, giving the connectivity measure something solid to work with.

**Change from template -- stimulus-locked connectivity window (0.0 to 0.6 s):**
The template computed connectivity across the full epoch. I restrict it to 0.0 to 0.6 seconds post-stimulus, which is the window where active cognitive processing is happening. This avoids contaminating the estimate with the pre-stimulus baseline or late activity that is unrelated to the task.

**Why we do this:**
This is the step where we actually measure what we set out to measure. All the previous steps were building up to producing clean, reliable input for exactly this computation.

In [ ]:
# Wrap ROI time series in an EpochsArray so mne-connectivity can process them
roi_ch_names     = [f"ROI{i+1}" for i in range(n_rois)]
info_roi         = mne.create_info(ch_names=roi_ch_names, sfreq=sfreq, ch_types="misc")
roi_epochs_fixed = mne.EpochsArray(label_ts_all, info_roi, tmin=ep_tmin, verbose=False)

# Compute ciPLV point estimate using all trials with multitaper
con = spectral_connectivity_epochs(
    roi_epochs_fixed,
    method="ciplv",        # corrected imaginary PLV: robust to volume conduction
    mode="multitaper",     # lower spectral variance than Fourier, better for short windows
    sfreq=sfreq,
    fmin=THETA_BAND[0],
    fmax=THETA_BAND[1],
    faverage=True,         # average across frequencies to get one value per ROI pair
    mt_adaptive=True,      # adapt taper weights to local SNR
    mt_low_bias=True,      # only use tapers with good spectral concentration
    mt_bandwidth=2.0,      # controls spectral smoothing in Hz
    tmin=CONN_TMIN,        # start at stimulus onset
    tmax=CONN_TMAX,        # end at 600 ms
    verbose=False
)

# Extract as a full connectivity matrix and remove the trailing frequency dimension
ci_fixed = np.squeeze(con.get_data(output="dense"))

print(f"ciPLV matrix shape: {ci_fixed.shape}")
print(f"ciPLV mean={ci_fixed.mean():.4f}  max={ci_fixed.max():.4f}")

---

### Step 8 -- Surrogate Resampling for Trial-Count Balancing

**What it does:**
This step runs the connectivity analysis not once, but 500 times. Each time it uses a randomly selected subset of trials. The 500 resulting connectivity matrices are then averaged together to produce the final estimate for each subject.

**Change from template -- this entire step is new:**
The template had no trial balancing at all. It simply computed connectivity using whatever trials were available for each subject. The problem is that connectivity estimates are sensitive to how many trials go into them. A subject with 150 clean epochs will produce a different estimate than a subject with 80, not necessarily because their brain connectivity is different, but simply because of the different trial counts. In an aging study where artifact rejection rates vary across participants, this is a real confound.

To fix this, I set the subsample size to 94, which was the lowest trial count any subject had after artifact rejection. For each of the 500 draws, I randomly subsample exactly 94 trials from whatever is available for that subject. Averaging across 500 draws gives a stable estimate, and because every subject is evaluated on the same number of trials, the results are directly comparable.

**Why Fourier here and not multitaper:**
Each of the 500 draws needs to run quickly. Fourier is faster than multitaper, and since we are averaging 500 estimates together the stability that multitaper provides is already achieved through the averaging itself. Multitaper is reserved for the single point estimate in Step 7 where there is no averaging to fall back on.

**Why we do this:**
Without this step, differences in connectivity between subjects or conditions could partly reflect differences in how many trials were available rather than actual differences in brain activity. This step removes that confound and makes the final scores fair and comparable across everyone in the study.

In [ ]:
# Upper triangle indices -- we only compute each pair once (no self-connectivity on diagonal)
iu, ju = np.triu_indices(n_rois, k=1)

# Number of trials to draw per surrogate
# 94 was chosen because it was the lowest trial count across all subjects
n_draw = min(n_total, SUBSAMPLE_N)
if n_total < SUBSAMPLE_N:
    print(f"WARNING: {n_total} epochs available, which is less than subsample_n={SUBSAMPLE_N}. Using all trials.")

# Storage for all surrogate connectivity matrices
ci_surrogates = np.zeros((N_SURROGATES, n_rois, n_rois))

for s in range(N_SURROGATES):

    # Randomly pick n_draw trials without replacement
    pick_idx   = rng.choice(n_total, size=n_draw, replace=False)
    label_ts   = label_ts_all[pick_idx]
    roi_epochs = mne.EpochsArray(label_ts, info_roi, tmin=ep_tmin, verbose=False)

    # Suppress harmless MNE warnings that appear during surrogate iterations
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="There were no Annotations stored*")

        con = spectral_connectivity_epochs(
            roi_epochs,
            method="ciplv",
            mode="fourier",     # faster than multitaper; variance is handled by averaging 500 draws
            sfreq=sfreq,
            fmin=THETA_BAND[0],
            fmax=THETA_BAND[1],
            faverage=True,
            tmin=CONN_TMIN,
            tmax=CONN_TMAX,
            indices=(iu, ju),   # only compute upper triangle to save time
            verbose=False
        )

    # Reconstruct the full symmetric matrix from the upper triangle values
    ci_vals      = np.squeeze(con.get_data())
    mat          = np.zeros((n_rois, n_rois))
    mat[iu, ju]  = ci_vals
    mat[ju, iu]  = ci_vals  # mirror to lower triangle since ciPLV is symmetric
    ci_surrogates[s] = mat

    if (s + 1) % 100 == 0:
        print(f"  Surrogate {s+1}/{N_SURROGATES} done")

# Average across all 500 draws
ci_mean = ci_surrogates.mean(axis=0)
print(f"\nSurrogate-averaged ciPLV: mean={ci_mean.mean():.4f}  max={ci_mean.max():.4f}")

---

### Step 9 -- Fronto-Parietal Summary Statistics

**What it does:**
From the full ROI by ROI connectivity matrix, this step extracts three summary scores per subject: left hemisphere fronto-parietal connectivity, right hemisphere fronto-parietal connectivity, and the overall average of the two.

**Why we do this:**
The full connectivity matrix contains values for every possible ROI pair, but what we actually care about is the relationship between frontal and parietal regions specifically. These three summary scores are the dependent variables that get passed to the statistical analysis in R. Having LH and RH separately means we can test whether aging or task condition affects one hemisphere more than the other.

In [ ]:
# np.ix_ creates an index mesh for extracting a submatrix block
# for example ci_mean[np.ix_(idx_flh, idx_plh)] gives the LH frontal to LH parietal block

# Left hemisphere: average of frontal->parietal and parietal->frontal
lh_fp = np.mean([
    ci_mean[np.ix_(idx_flh, idx_plh)].mean(),
    ci_mean[np.ix_(idx_plh, idx_flh)].mean()
])

# Right hemisphere
rh_fp = np.mean([
    ci_mean[np.ix_(idx_frh, idx_prh)].mean(),
    ci_mean[np.ix_(idx_prh, idx_frh)].mean()
])

# Overall mean across both hemispheres
fp_mean = np.mean([lh_fp, rh_fp])

result = {
    "sub":       SUBJECT_ID,
    "condition": CONDITION,
    "n_epochs":  n_total,
    "lh_fp":     float(lh_fp),
    "rh_fp":     float(rh_fp),
    "fp_mean":   float(fp_mean)
}

print(f"Subject {SUBJECT_ID} | condition: {CONDITION}")
print(f"  n_epochs = {n_total}")
print(f"  lh_fp    = {lh_fp:.4f}")
print(f"  rh_fp    = {rh_fp:.4f}")
print(f"  fp_mean  = {fp_mean:.4f}")

---

### Step 10 -- Save Results

**What it does:**
The results are saved in two formats. JSON stores the full output including ROI names, condition, and surrogate settings alongside the connectivity values. CSV saves a flat table that can be directly imported into R or Python for statistical analysis.

**Why we do this:**
The CSV format matches the output structure of the full multi-subject pipeline, so a single-subject run here is immediately compatible with the group-level analysis scripts. The JSON is useful for archiving and reproducibility since it captures all the metadata alongside the numbers.

In [ ]:
# Save full metadata and results as JSON
full_results = {
    "condition":    CONDITION,
    "fp_summary":   [result],
    "n_surrogates": N_SURROGATES,
    "roi_names":    roi_names
}

json_path = OUT_DIR / f"fp_ciPLV_sub{SUBJECT_ID}_{CONDITION}.json"
with open(json_path, "w") as f:
    json.dump(full_results, f, indent=2)
print(f"Saved: {json_path}")

# Save as CSV -- compatible with group-level pipeline output
csv_path = OUT_DIR / f"fp_summary_sub{SUBJECT_ID}_{CONDITION}.csv"
df = pd.DataFrame([result])
df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")

print("\n=== DONE ===")
print(df.to_string(index=False))

---

## Run -- Single Subject Test

*Note: The main block below was generated by AI (Claude) and was used solely to test whether the pipeline runs correctly on a single subject. It is not part of the original analysis code.*

In [ ]:
if __name__ == "__main__":

    # Update these two lines to point to your data before running
"    EEGLAB_DIR = Path("../data")  # folder containing the .set file
"
    SUBJECT_ID = 301                          # subject ID to test
"    CONDITION  = "offtask"                     # ontask or offtask
"

    print(f"Running pipeline for subject {SUBJECT_ID}, condition: {CONDITION}")
    print(f"Data directory: {EEGLAB_DIR}")
    print("=" * 50)

    # Check that the file exists before running
    file_pattern = f"sub-{{sub}}_erpcon_stimlocked_{CONDITION}.set"
    fname = EEGLAB_DIR / file_pattern.format(sub=SUBJECT_ID)

    if not fname.exists():
        print(f"ERROR: File not found at {fname}")
        print("Please check EEGLAB_DIR and SUBJECT_ID above.")
    else:
        print(f"File found: {fname}")
        print("All steps above have already been run in order.")
        print("Results saved to the output folder.")
        print(f"\nFinal results for subject {SUBJECT_ID}:")
        print(df.to_string(index=False))